# Checkpoint 8: assemble the first training table

Each row will describe one shipment at one decision time. Run this notebook from the top with `.venv`.

We use the 48-hour reporting allowance chosen in checkpoint 7. It is a completeness assumption, not a guaranteed reporting deadline. The 70% cutoff is still an example for learning; notebook 11 defines the final evaluation split.

The table keeps three things separate:

- **X:** features calculated from information available at the decision time.
- **y:** the later incident outcome, known by the training cutoff and old enough to use.
- **Metadata:** IDs, timestamps and audit details that help trace the row.

The next cells list the nine feature columns explicitly. No model is trained here, and the three-hour lookback has not been optimized.

In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from statistics import mean
import json
import math
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/events.jsonl").is_file()
             and (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from inside the candidate repository.")

def load(name):
    with (ROOT / "data" / name).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

raw_events = load("events.jsonl")
decisions = pd.DataFrame(load("decision_times.jsonl"))
reports = pd.DataFrame(load("labels.jsonl"))
decisions["decision_time"] = pd.to_datetime(decisions["decision_time"], utc=True)
for field in ("incident_at", "label_available_at"):
    reports[field] = pd.to_datetime(reports[field], utc=True)
HORIZON = pd.Timedelta(hours=6)
GRACE_HOURS = 48
WINDOW_HOURS = 3


## 1. Reuse the definitions we already explored

The next cell contains unchanged copies of the helpers from checkpoints 2–4 and 7. They are included here so this notebook does not rely on another notebook's execution state. They do not run other notebooks or load their saved outputs.

When we move these experiments into Python modules, these helpers will become shared functions with one maintained implementation. For now their limitations remain: provisional clock rules, no bounded online state, and assumed outcome completeness.


In [2]:
def utc(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if value.tzinfo is None:
        raise ValueError("An explicit timezone is required.")
    return value.astimezone(timezone.utc)

def known_revisions(records, checkpoint):
    """Select the highest revision received by the checkpoint for each event ID."""
    if checkpoint.tzinfo is None:
        raise ValueError("Checkpoint must include a timezone")
    checkpoint = checkpoint.astimezone(timezone.utc)
    delivered = {}
    latest = {}
    for event in records:
        if utc(event["received_at"]) > checkpoint:
            continue
        key = (event["event_id"], event["revision"])
        # Compare canonical timestamps so equivalent timezone notation agrees.
        normalized = dict(event)
        for field in ("device_time", "received_at"):
            normalized[field] = utc(event[field]).isoformat()
        signature = json.dumps(normalized, sort_keys=True, allow_nan=False)
        if key in delivered:
            if delivered[key] != signature:
                raise ValueError("Conflicting content for the same event/revision")
            continue
        delivered[key] = signature
        prior = latest.get(event["event_id"])
        if prior is not None and prior["shipment_id"] != event["shipment_id"]:
            raise ValueError("An event ID changed shipment")
        if prior is None or event["revision"] > prior["revision"]:
            latest[event["event_id"]] = dict(event)
    return [latest[event_id] for event_id in sorted(latest)]

def first_features(records, shipment, checkpoint):
    selected = known_revisions(records, checkpoint)
    usable = []
    for event in selected:
        if event["shipment_id"] != shipment or event["kind"] != "temperature_c":
            continue
        value = event["value"]
        if isinstance(value, bool) or not isinstance(value, (float, int)):
            continue
        if not math.isfinite(value):
            continue
        measured = utc(event["device_time"])
        received = utc(event["received_at"])
        if measured > received:  # Exclude measurements whose clocks run ahead of receipt.
            continue
        usable.append(event)
    if not usable:
        return {"latest_temperature_c": None, "measurement_age_minutes": None,
                "arrival_delay_minutes": None, "temperature_missing": 1}
    latest = max(usable, key=lambda e: (utc(e["device_time"]),
                                      utc(e["received_at"]), e["event_id"]))
    measured = utc(latest["device_time"])
    received = utc(latest["received_at"])
    return {
        "latest_temperature_c": float(latest["value"]),
        "measurement_age_minutes": (checkpoint - measured).total_seconds() / 60,
        "arrival_delay_minutes": (received - measured).total_seconds() / 60,
        "temperature_missing": 0,
    }

def window_features(records, shipment, checkpoint, window_hours=3):
    if checkpoint.tzinfo is None:
        raise ValueError("Checkpoint needs an explicit timezone")
    if not math.isfinite(window_hours) or window_hours <= 0:
        raise ValueError("Window must be finite and positive")
    left = checkpoint - timedelta(hours=window_hours)
    points = []
    for event in known_revisions(records, checkpoint):
        if event["shipment_id"] != shipment or event["kind"] != "temperature_c":
            continue
        value = event["value"]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            continue
        measured, received = utc(event["device_time"]), utc(event["received_at"])
        # Use the same clock rule as the latest-temperature feature.
        if measured > received or not left < measured <= checkpoint:
            continue
        points.append((measured, event["event_id"], float(value)))
    points.sort(key=lambda p: (p[0], p[1]))
    if not points:
        return {"temperature_count": 0, "temperature_mean_c": None,
                "temperature_max_c": None, "temperature_trend_c_per_hour": None,
                "temperature_span_hours": None}
    values = [p[2] for p in points]
    hours = [(p[0] - points[0][0]).total_seconds() / 3600 for p in points]
    x_mean, y_mean = mean(hours), mean(values)
    denominator = sum((x - x_mean)**2 for x in hours)
    slope = (sum((x - x_mean)*(y - y_mean) for x, y in zip(hours, values))
             / denominator) if denominator > 0 else None
    return {"temperature_count": len(points), "temperature_mean_c": y_mean,
            "temperature_max_c": max(values), "temperature_trend_c_per_hour": slope,
            "temperature_span_hours": hours[-1]}

def eligibility_table(checkpoints, incident_reports, training_cutoff, grace_hours=48):
    if training_cutoff.tzinfo is None:
        raise ValueError("Training cutoff must be timezone-aware")
    if grace_hours < 0:
        raise ValueError("Reporting allowance cannot be negative")
    grace = pd.Timedelta(hours=grace_hours)
    # Future report contents do not participate in historical label construction.
    known = incident_reports.loc[incident_reports["label_available_at"] <= training_cutoff]
    grouped = {sid: group for sid, group in known.groupby("shipment_id", sort=False)}
    result = []
    for row in checkpoints.itertuples(index=False):
        end = row.decision_time + HORIZON
        eligible_at = end + grace
        label = None
        if row.decision_time >= training_cutoff:
            status = "outside_training_period"
        elif eligible_at > training_cutoff:
            status = "waiting_for_maturity"
        else:
            group = grouped.get(row.shipment_id)
            positive = group is not None and bool(((group["incident_at"] > row.decision_time)
                                                   & (group["incident_at"] <= end)).any())
            label = int(positive)
            status = "eligible_positive" if positive else "eligible_assumed_negative"
        result.append({"shipment_id": row.shipment_id, "decision_time": row.decision_time,
                       "horizon_end": end, "eligible_at": eligible_at,
                       "training_cutoff": training_cutoff, "grace_hours": grace_hours,
                       "status": status, "label": label})
    output = pd.DataFrame(result)
    output["label"] = output["label"].astype("Int64")
    return output

In [3]:
unique_times = sorted(decisions["decision_time"].unique())
cutoff = pd.Timestamp(unique_times[min(int(len(unique_times)*0.70), len(unique_times)-1)])
all_statuses = eligibility_table(decisions, reports, cutoff, GRACE_HOURS)
eligible = all_statuses.loc[all_statuses["label"].notna()].copy()
events_by_shipment = {}
for event in raw_events:
    events_by_shipment.setdefault(event["shipment_id"], []).append(event)

FEATURE_COLUMNS = [
    "latest_temperature_c", "measurement_age_minutes", "arrival_delay_minutes",
    "temperature_missing", "temperature_count", "temperature_mean_c",
    "temperature_max_c", "temperature_trend_c_per_hour", "temperature_span_hours",
]
rows = []
for item in eligible.itertuples(index=False):
    checkpoint = item.decision_time.to_pydatetime()
    history = events_by_shipment.get(item.shipment_id, [])
    features = first_features(history, item.shipment_id, checkpoint)
    features.update(window_features(history, item.shipment_id, checkpoint, WINDOW_HOURS))
    rows.append({"shipment_id": item.shipment_id, "decision_time": item.decision_time,
                 **features, "label": int(item.label), "horizon_end": item.horizon_end,
                 "eligible_at": item.eligible_at, "training_cutoff": cutoff,
                 "grace_hours": GRACE_HOURS, "lookback_hours": WINDOW_HOURS})
training_table = pd.DataFrame(rows).sort_values(["decision_time", "shipment_id"]).reset_index(drop=True)
X = training_table[FEATURE_COLUMNS].copy()
y = training_table["label"].copy()
print("Training cutoff:", cutoff.isoformat())
print("Examples:", len(training_table), "Candidate feature columns:", len(FEATURE_COLUMNS))
print("Positive:", int(y.sum()), "Assumed negative:", int((y == 0).sum()))
display(training_table.head(8))


Training cutoff: 2026-02-22T23:00:00+00:00
Examples: 1209 Candidate feature columns: 9
Positive: 96 Assumed negative: 1113


,shipment_id,decision_time,latest_temperature_c,measurement_age_minutes,arrival_delay_minutes,temperature_missing,temperature_count,temperature_mean_c,temperature_max_c,temperature_trend_c_per_hour,temperature_span_hours,label,horizon_end,eligible_at,training_cutoff,grace_hours,lookback_hours
0,s-00000,2026-01-01 08:00:00+00:00,3.798,60.0,35.0,0,2,4.242500,4.687,-0.8890,1.0,0,2026-01-01 14:00:00+00:00,2026-01-03 14:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
1,s-00000,2026-01-01 11:00:00+00:00,5.386,60.0,2.0,0,1,5.386000,5.386,NaN,0.0,1,2026-01-01 17:00:00+00:00,2026-01-03 17:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
2,s-00001,2026-01-01 11:00:00+00:00,4.436,60.0,0.0,0,2,4.060000,4.436,0.7520,1.0,0,2026-01-01 17:00:00+00:00,2026-01-03 17:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
3,s-00000,2026-01-01 14:00:00+00:00,9.857,0.0,0.0,0,3,8.977667,9.857,1.1125,2.0,1,2026-01-01 20:00:00+00:00,2026-01-03 20:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
4,s-00001,2026-01-01 14:00:00+00:00,6.839,0.0,0.0,0,3,5.270000,6.839,1.0275,2.0,0,2026-01-01 20:00:00+00:00,2026-01-03 20:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
5,s-00002,2026-01-01 14:00:00+00:00,4.184,60.0,2.0,0,2,4.517000,4.850,-0.6660,1.0,0,2026-01-01 20:00:00+00:00,2026-01-03 20:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
6,s-00001,2026-01-01 17:00:00+00:00,7.380,60.0,2.0,0,1,7.380000,7.380,NaN,0.0,1,2026-01-01 23:00:00+00:00,2026-01-03 23:00:00+00:00,2026-02-22 23:00:00+00:00,48,3
7,s-00002,2026-01-01 17:00:00+00:00,4.325,60.0,2.0,0,2,4.406000,4.487,-0.1620,1.0,0,2026-01-01 23:00:00+00:00,2026-01-03 23:00:00+00:00,2026-02-22 23:00:00+00:00,48,3


## 2. Read one row in plain English

Choose a positive row for illustration, without using its outcome to calculate its features. Its feature values describe the earlier checkpoint; its positive label describes an audited incident that occurred later within six hours.

The model will eventually learn from many such rows. It will not be given the label as an input for predicting a new shipment.


In [4]:
example = training_table.loc[training_table["label"] == 1].iloc[0]
print("Shipment:", example["shipment_id"])
print("Original prediction time:", example["decision_time"])
display(example[FEATURE_COLUMNS].to_frame("feature_value"))
print("Eventual training label:", example["label"])
matched = reports.loc[(reports["shipment_id"] == example["shipment_id"])
                      & (reports["incident_at"] > example["decision_time"])
                      & (reports["incident_at"] <= example["horizon_end"])
                      & (reports["label_available_at"] <= cutoff)]
display(matched[["incident_at", "label_available_at"]])
print("Incident details shown only for explanation; they are absent from X.")


Shipment: s-00000
Original prediction time: 2026-01-01 11:00:00+00:00


,feature_value
latest_temperature_c,5.386
measurement_age_minutes,60.0
arrival_delay_minutes,2.0
temperature_missing,0
temperature_count,1
temperature_mean_c,5.386
temperature_max_c,5.386
temperature_trend_c_per_hour,NaN
temperature_span_hours,0.0


Eventual training label: 1


,incident_at,label_available_at
0,2026-01-01 17:00:00+00:00,2026-01-02 11:00:00+00:00


Incident details shown only for explanation; they are absent from X.


## 3. Inspect missing inputs and the outcome balance

Some windows contain insufficient evidence for a slope. Do not fill it with zero automatically. Model-specific missing-value handling will be learned using training data only in the next experiment.

If most examples are negative, accuracy alone can be misleading: a model that always says negative can appear accurate while missing every incident. We will learn baseline probability predictions and evaluation metrics before making performance claims.

These nine features are candidates. They may be redundant; more columns do not automatically improve predictions.


In [5]:
display(X.isna().sum().rename("missing_rows").to_frame())
display(y.value_counts().sort_index().rename_axis("label").rename("rows").to_frame())
print("Positive fraction in this eligible training subset:", round(float(y.mean()), 4))
assert not training_table.duplicated(["shipment_id", "decision_time"]).any()
assert training_table["eligible_at"].le(cutoff).all()
assert len(X) == len(y) == len(eligible)
assert "label" not in X.columns and "shipment_id" not in X.columns
assert "training_cutoff" not in X.columns

# Adding information unavailable at a decision must not change its features.
probe = training_table.iloc[0]
when = probe["decision_time"].to_pydatetime()
history = events_by_shipment[probe["shipment_id"]]
future = {"event_id": "checkpoint8-future-probe", "revision": 1,
          "shipment_id": probe["shipment_id"], "device_time": when.isoformat(),
          "received_at": (when + timedelta(days=1)).isoformat(),
          "kind": "temperature_c", "value": 999.0, "source": "test", "payload": {}}
for feature_fn in (first_features, window_features):
    assert feature_fn(history, probe["shipment_id"], when) == feature_fn(history+[future], probe["shipment_id"], when)
assert eligibility_table(decisions, reports.loc[reports["label_available_at"] <= cutoff], cutoff).equals(all_statuses)
print("Passed: maturity, explicit feature separation, row uniqueness, and unavailable information checks.")


,missing_rows
latest_temperature_c,0
measurement_age_minutes,0
arrival_delay_minutes,0
temperature_missing,0
temperature_count,0
temperature_mean_c,19
temperature_max_c,19
temperature_trend_c_per_hour,316
temperature_span_hours,19


,rows
label,
0,1113
1,96


Positive fraction in this eligible training subset: 0.0794
Passed: maturity, explicit feature separation, row uniqueness, and unavailable information checks.


## 4. Save the derived training table for inspection

This is an experimental training subset under the stated cutoff and assumptions, not the entire dataset and not a final submission artifact. The original source JSON remains authoritative. Re-running regenerates the export. CSV does not preserve pandas types; timestamps need explicit UTC parsing when reloaded.


In [6]:
output_dir = ROOT / 'personal' / 'data' / 'tables'
output_dir.mkdir(parents=True, exist_ok=True)
output = output_dir / "checkpoint08_training_rows.csv"
training_table.to_csv(output, index=False, lineterminator="\n")
loaded = pd.read_csv(output)
assert len(loaded) == len(training_table)
assert loaded["shipment_id"].tolist() == training_table["shipment_id"].tolist()
assert loaded["label"].tolist() == y.tolist()
print("Saved:", output.relative_to(ROOT))


Saved: data/tables/checkpoint08_training_rows.csv


## 5. What happens next?

Next we explain a **baseline**: a simple prediction to beat. Then we learn how logistic regression turns weighted features into probabilities, how training adjusts those weights, and how it differs from a decision tree. Before fitting preprocessing or choosing models, define chronological validation/test periods and when their outcomes are considered complete. The existing 70% demonstration cutoff is not a final split decision.

**Interview notes:** “Each row represents one decision checkpoint. I calculated inputs from information available then, attached mature labels known by fit time, and explicitly separated feature columns from outcome and audit metadata.”

**Try explaining this:** why must the `label` column be excluded from X, even though it appears in the same training table?
